# Spectrum-SLM — Cognitive Radio Spectrum Sensing
### Production-Grade Transformer Training on Kaggle GPU

**Authors:** Anjani · Ashish Joshi · Mayank  
**Guide:** Dr. Abhinandan S.P.  
**Repo:** https://github.com/31ASHISH/Spectrum-SLM

---

**Before running:**
1. Set **Accelerator = GPU T4 x2** (Settings → Accelerator)
2. Enable **Internet** (Settings → Internet → On)
3. Set **Persistence = Files** (Settings → Persistence)
4. Your SDR dataset must be added as a Kaggle Dataset input

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Clone Spectrum-SLM code from GitHub
# ═══════════════════════════════════════════════════════════════

!git clone https://github.com/31ASHISH/Spectrum-SLM.git /kaggle/working/Spectrum-SLM
print("Repo cloned successfully!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Move into project directory
# ═══════════════════════════════════════════════════════════════

%cd /kaggle/working/Spectrum-SLM
!ls -la

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Install dependencies
# ═══════════════════════════════════════════════════════════════

!pip install -q scikit-learn pandas tqdm
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Configure Paths
#
# IMPORTANT: Edit DATA_ROOT below to match your Kaggle dataset path.
# After adding your dataset, check /kaggle/input/ to find the folder name.
# ═══════════════════════════════════════════════════════════════

import os, sys
sys.path.insert(0, '/kaggle/working/Spectrum-SLM')

# ── Edit this path to match your Kaggle dataset name ─────────────────────────
DATA_ROOT = '/kaggle/input/spectrum-slm-sdr-data'   # <-- change if needed
# ─────────────────────────────────────────────────────────────────────────────

SU_DIR   = f'{DATA_ROOT}/Secondary_User'
NEW_DIR  = f'{DATA_ROOT}/files-20260414T094743Z-3-001'
OUT_DIR  = '/kaggle/working/checkpoints'

os.makedirs(f'{OUT_DIR}/phase1', exist_ok=True)
os.makedirs(f'{OUT_DIR}/phase2', exist_ok=True)

# Verify paths
print('=== Path Check ===')
for name, path in [('DATA_ROOT', DATA_ROOT), ('SU_DIR', SU_DIR), ('NEW_DIR', NEW_DIR)]:
    exists = os.path.isdir(path)
    status = '✓' if exists else '✗ NOT FOUND'
    print(f'  {name}: {path}  [{status}]')

# List input datasets
print('\n=== /kaggle/input/ contents ===')
for d in os.listdir('/kaggle/input'):
    print(f'  {d}/')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Override config.py for Kaggle paths
# ═══════════════════════════════════════════════════════════════

import config

config.SECONDARY_USER_DIR = SU_DIR
config.NEW_DATASET_DIR    = NEW_DIR
config.PHASE1_DATA_FILE   = f'{SU_DIR}/psd_binned_by_snr_.pth'
config.CKPT_ROOT          = OUT_DIR
config.CKPT_PHASE1        = f'{OUT_DIR}/phase1'
config.CKPT_PHASE2        = f'{OUT_DIR}/phase2'
config.PHASE1_DATA_DIR    = SU_DIR
config.PHASE2_DATA_DIR    = NEW_DIR

print('=== Spectrum-SLM Config (Kaggle) ===')
print(f'  N_BINS         : {config.N_BINS}')
print(f'  SEQ_LEN        : {config.SEQ_LEN}  (192 tokens + 1 CLS)')
print(f'  N_MOD_CLASSES  : {config.N_MOD_CLASSES_V2}  {config.MOD_NAMES_V2}')
print(f'  CKPT_PHASE1    : {config.CKPT_PHASE1}')
print(f'  CKPT_PHASE2    : {config.CKPT_PHASE2}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Dataset verification
# ═══════════════════════════════════════════════════════════════

import numpy as np
from dataset.loader import load_all_real_data

print('Loading all real SDR data...')
psds, pu, mod, snr = load_all_real_data(SU_DIR, NEW_DIR)

print(f'\n=== Dataset Summary ===')
print(f'  Total samples : {len(psds):,}')
print(f'  PSD shape     : {psds.shape}   <- must be (N, 192)')
print(f'  NaN count     : {np.isnan(psds).sum()}')
print(f'  PU=1 (active) : {pu.sum():,} ({pu.mean()*100:.1f}%)')
print(f'  SNR range     : {snr.min():.1f} - {snr.max():.1f} dB')
print(f'\n  Modulation breakdown:')
names = {0:'BPSK', 1:'QPSK', 2:'8PSK', 3:'16QAM', 4:'DQPSK'}
for m in range(5):
    n = int((mod==m).sum())
    print(f'    {names[m]:<8}: {n:>7,}')

assert psds.shape[1] == 192, f'ERROR: Expected 192 bins, got {psds.shape[1]}'
assert np.isnan(psds).sum() == 0, 'ERROR: NaN values found in PSDs!'
print('\nDataset verification: PASSED')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Architecture Sanity Check
# ═══════════════════════════════════════════════════════════════

import torch
from spectrum_slm_model import SpectrumSLM

model = SpectrumSLM(n_bins=192, patch_size=1, d_model=128,
                    nhead=4, num_layers=4, dim_feedforward=512,
                    dropout=0.1, n_mod_classes=5)

print(f'=== Model Architecture ===')
print(f'  Parameters      : {model.count_parameters():,}')
print(f'  Input           : PSD (B, 192)')
print(f'  Token sequence  : 193  (192 bin-tokens + 1 CLS)')
print(f'  Modulation cls  : 5  (BPSK/QPSK/8PSK/16QAM/DQPSK)')

# Shape verification
psd_dummy = torch.randn(4, 192)
out = model(psd_dummy, return_msm=True)
print(f'\n  Output shapes:')
for k, v in out.items():
    print(f'    {k:<12}: {v.shape}')

assert out['pu_logits'].shape  == (4, 2)
assert out['mod_logits'].shape == (4, 5)
assert out['snr_pred'].shape   == (4,)
assert out['msm_pred'].shape   == (4, 192, 1)
print('\nArchitecture check: PASSED')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Generate Dataset Reports
# ═══════════════════════════════════════════════════════════════

from dataset.analysis import run_analysis

report = run_analysis(out_dir='/kaggle/working')
print('\nReports saved to /kaggle/working/')

---
## Phase 1 — Masked Spectrum Modelling (MSM Pre-training)

- Self-supervised pre-training on `psd_binned_by_snr_.pth` (17,360 real samples)
- Randomly masks 20% of 192 frequency-bin tokens
- Learns to reconstruct masked bins
- Estimated time: **~25 min on T4 GPU**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — PHASE 1: Masked Spectrum Modelling
# ═══════════════════════════════════════════════════════════════

from training.train_phase1 import run_phase1

history_p1 = run_phase1(
    data_dir   = SU_DIR,
    save_dir   = f'{OUT_DIR}/phase1',
    epochs     = 30,
    lr         = 3e-4,
    batch_size = 64,
    patience   = 5,
    resume     = True,   # auto-resume if checkpoint exists
)

best_msm = min(h['val_msm'] for h in history_p1) if history_p1 else 'N/A'
print(f'\nPhase 1 complete.  Best Val MSM Loss: {best_msm}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — Plot Phase 1 Training Curve
# ═══════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

if history_p1:
    epochs_p1 = [h['epoch']     for h in history_p1]
    tr_msm    = [h['train_msm'] for h in history_p1]
    vl_msm    = [h['val_msm']   for h in history_p1]

    plt.figure(figsize=(10, 4))
    plt.plot(epochs_p1, tr_msm, label='Train MSM Loss', color='#58a6ff', linewidth=2)
    plt.plot(epochs_p1, vl_msm, label='Val MSM Loss',   color='#f78166', linewidth=2, linestyle='--')
    plt.xlabel('Epoch'); plt.ylabel('MSM Loss')
    plt.title('Phase 1 — Masked Spectrum Modelling Training Curve')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/kaggle/working/phase1_training_curve.png', dpi=150)
    plt.show()
    print('Saved: /kaggle/working/phase1_training_curve.png')

---
## Phase 2 — Supervised Multi-task Fine-tuning

- **PU Detection** (Binary: present/absent) — Focal Loss
- **Modulation Classification** (5-class: BPSK/QPSK/8PSK/16QAM/DQPSK) — CrossEntropy
- **SNR Estimation** (regression, dB) — HuberLoss
- Kendall uncertainty weighting for multi-task balance
- Estimated time: **~60 min on T4 GPU**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — PHASE 2: Supervised Multi-task Fine-tuning
# ═══════════════════════════════════════════════════════════════

from training.train_phase2 import run_phase2

metrics = run_phase2(
    secondary_user_dir = SU_DIR,
    new_dataset_dir    = NEW_DIR,
    save_dir           = f'{OUT_DIR}/phase2',
    epochs             = 50,
    batch_size         = 64,
    lr                 = 1e-4,
    patience           = 8,
    learn_weights      = True,   # Kendall uncertainty weighting
    resume             = True,   # auto-resume from checkpoint
    resume_phase1      = True,   # init from Phase 1 weights
)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12 — Final Results Summary
# ═══════════════════════════════════════════════════════════════

print('=' * 60)
print('  SPECTRUM-SLM — FINAL TEST SET RESULTS')
print('=' * 60)
print(f'\n  PU Detection')
print(f'    Accuracy      : {metrics["pu_accuracy"]*100:.2f}%')
print(f'    F1 Score      : {metrics["pu_f1"]:.4f}')
print(f'    ROC-AUC       : {metrics["pu_auc"]:.4f}')
print(f'    PR-AUC        : {metrics["pu_pr_auc"]:.4f}')
print(f'    Low-SNR Acc   : {metrics["low_snr_pu_acc"]*100:.2f}%  (<8 dB)')
print(f'    Low-SNR F1    : {metrics["low_snr_pu_f1"]:.4f}')
print(f'\n  Modulation Classification')
print(f'    Accuracy      : {metrics["mod_accuracy"]*100:.2f}%')
print(f'    Macro F1      : {metrics["mod_f1_macro"]:.4f}')
print(f'\n  SNR Estimation')
print(f'    MAE           : {metrics["snr_mae_db"]:.3f} dB')
print(f'    RMSE          : {metrics["snr_rmse_db"]:.3f} dB')
print(f'    R-squared     : {metrics["snr_r2"]:.4f}')
print(f'\n  Samples evaluated: {metrics["n_samples"]:,}')
print('=' * 60)

print('\n  Per-SNR Bin Breakdown:')
print(f'  {"SNR(dB)":<8} {"PU Acc":<10} {"PU F1":<10} {"SNR MAE":<10} {"n"}')
print('  ' + '-'*45)
for snr_b, m in sorted(metrics['per_snr_metrics'].items(), key=lambda x: int(x[0])):
    print(f'  {snr_b:>4} dB   {m["pu_acc"]*100:>6.1f}%   {m["pu_f1"]:>6.3f}    {m["snr_mae"]:>6.3f} dB  {m["n"]:>5}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13 — Training Curves (Phase 2)
# ═══════════════════════════════════════════════════════════════

import json
import matplotlib.pyplot as plt

hist_path = f'{OUT_DIR}/phase2/training_history_phase2.json'
if os.path.exists(hist_path):
    with open(hist_path) as f:
        history_p2 = json.load(f)

    epochs_p2 = [h['epoch']       for h in history_p2]
    tr_total  = [h['train_total'] for h in history_p2]
    vl_total  = [h['val_total']   for h in history_p2]
    tr_pu     = [h['train_pu']    for h in history_p2]
    tr_mod    = [h['train_mod']   for h in history_p2]
    tr_snr    = [h['train_snr']   for h in history_p2]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Total loss
    axes[0].plot(epochs_p2, tr_total, label='Train Total', color='#58a6ff', lw=2)
    axes[0].plot(epochs_p2, vl_total, label='Val Total',   color='#f78166', lw=2, ls='--')
    axes[0].set_title('Phase 2 — Total Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Task breakdown
    axes[1].plot(epochs_p2, tr_pu,  label='PU (Focal)',   color='#3fb950', lw=2)
    axes[1].plot(epochs_p2, tr_mod, label='Mod (CE)',     color='#ffa657', lw=2)
    axes[1].plot(epochs_p2, tr_snr, label='SNR (Huber)', color='#d2a8ff', lw=2)
    axes[1].set_title('Phase 2 — Per-Task Train Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/phase2_training_curves.png', dpi=150)
    plt.show()
    print('Saved: /kaggle/working/phase2_training_curves.png')
else:
    print(f'History not found at {hist_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 14 — Confusion Matrix (PU Detection)
# ═══════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt

cm = np.array(metrics['pu_confusion'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred: Idle','Pred: Active'])
ax.set_yticklabels(['True: Idle','True: Active'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=14, color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_title('PU Detection — Confusion Matrix')
plt.tight_layout()
plt.savefig('/kaggle/working/pu_confusion_matrix.png', dpi=150)
plt.show()
print('Saved: /kaggle/working/pu_confusion_matrix.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 15 — Per-SNR Accuracy Bar Chart
# ═══════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

snr_bins  = sorted(metrics['per_snr_metrics'].keys(), key=int)
pu_accs   = [metrics['per_snr_metrics'][b]['pu_acc']*100 for b in snr_bins]
snr_maes  = [metrics['per_snr_metrics'][b]['snr_mae']    for b in snr_bins]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(snr_bins, pu_accs, color='#58a6ff', edgecolor='white', linewidth=0.5)
axes[0].axhline(90, color='#f78166', linestyle='--', label='90% target')
axes[0].set_xlabel('SNR (dB)'); axes[0].set_ylabel('PU Accuracy (%)')
axes[0].set_title('PU Accuracy vs SNR Bin')
axes[0].set_ylim(0, 105); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(snr_bins, snr_maes, color='#3fb950', edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('SNR (dB)'); axes[1].set_ylabel('SNR MAE (dB)')
axes[1].set_title('SNR Estimation Error vs SNR Bin')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Per-SNR-Bin Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/per_snr_performance.png', dpi=150)
plt.show()
print('Saved: /kaggle/working/per_snr_performance.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 16 — Export ONNX Model
# ═══════════════════════════════════════════════════════════════

from training.export_onnx import export_onnx

export_onnx(
    ckpt_path = f'{OUT_DIR}/phase2/slm_phase2_best.pt',
    save_path = '/kaggle/working/spectrum_slm.onnx',
)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 17 — List all Output Files (ready to download)
# ═══════════════════════════════════════════════════════════════

import os

print('=' * 60)
print('  OUTPUT FILES  (download from the Output tab)')
print('=' * 60)
total_mb = 0
for root, dirs, files in os.walk('/kaggle/working'):
    # Skip hidden and git dirs
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size  = os.path.getsize(fpath) / 1e6
        total_mb += size
        rel = fpath.replace('/kaggle/working/', '')
        print(f'  {rel:<55} {size:>7.2f} MB')
print(f'\n  Total: {total_mb:.1f} MB')
print('\nDone! Download the files above from the Output tab.')